In [12]:
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import pandas as pd



In [13]:
xgdf_train = pd.read_parquet('/Users/Georgi/Dropbox/File group 4B/train_df_20241211.parquet', engine='pyarrow')
xgdf_test = pd.read_parquet('/Users/Georgi/Dropbox/File group 4B/test_df_20241211.parquet', engine='pyarrow')
xgdf_val = pd.read_parquet('/Users/Georgi/Dropbox/File group 4B/val_df_20241211.parquet', engine='pyarrow')

xgdf = pd.concat([xgdf_train, xgdf_test, xgdf_val], axis=0, ignore_index=True)

In [14]:
xgdf_train.head()

,store_nbr,item_nbr,date,onpromotion,store_type,store_cluster,item_family,item_class,perishable,year,week_number_cum,unit_sales
0,1,105857,2014-08-18,0,D,13,GROCERY I,1092,0,2014,86,24.000000
1,1,105857,2014-08-25,0,D,13,GROCERY I,1092,0,2014,87,56.000000
2,1,105857,2014-09-01,0,D,13,GROCERY I,1092,0,2014,88,44.000000
3,1,105857,2014-09-08,0,D,13,GROCERY I,1092,0,2014,89,52.000000
4,1,105857,2014-09-15,0,D,13,GROCERY I,1092,0,2014,90,48.785713


In [15]:
# Define the specific item and store numbers you're interested in
# item_number = 105857
# store_number = 1

# Filter the dataset
# filtered_df = xgdf[(xgdf['store_nbr'] == store_number)]
filtered_df = xgdf
# Show the filtered dataset
print(filtered_df)


         store_nbr  item_nbr       date  onpromotion store_type  \
0                1    105857 2014-08-18            0          D   
1                1    105857 2014-08-25            0          D   
2                1    105857 2014-09-01            0          D   
3                1    105857 2014-09-08            0          D   
4                1    105857 2014-09-15            0          D   
...            ...       ...        ...          ...        ...   
1988943          1   1324670 2017-07-10            0          D   
1988944          1   1324670 2017-07-17            0          D   
1988945          1   1324670 2017-07-24            0          D   
1988946          1   1324670 2017-07-31            0          D   
1988947          1   1324670 2017-08-07            0          D   

         store_cluster item_family  item_class  perishable  year  \
0                   13   GROCERY I        1092           0  2014   
1                   13   GROCERY I        1092           0 

In [16]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Assuming you have the dataset in `df`
# Set 'date' as index
filtered_df['date'] = pd.to_datetime(filtered_df['date'])
df = filtered_df.set_index('date')

# Predefined parameters
alpha = 0.5
beta = 0.3
gamma = 0.2
seasonal_periods = 52  # Assuming weekly seasonality
split_week = 189  # Cumulative week number for train-test split

# Initialize list to store RMSE results and predictions
rmse_results = []
all_predictions = []

# Group by store and item
for (store_nbr, item_nbr), group in df.groupby(['store_nbr', 'item_nbr']):
    # Split the data based on cumulative week number (<= 189 for train, > 189 for test)
    train = group[group['week_number_cum'] <= split_week]  # Train set: weeks <= 189
    test = group[(group['week_number_cum'] > split_week) & (group['week_number_cum'] <= 215)]
    # Test set: weeks > 189

    # Initialize storage for t+2 predictions
    t_plus_2_predictions = []
    test_dates = test.index

    # Rolling training data
    rolling_train = train.copy()

    # Iteratively forecast t+2
    for i in range(len(test)):
        # Fit the model on the rolling training data
        model = ExponentialSmoothing(
            rolling_train['unit_sales'],
            seasonal_periods=seasonal_periods,
            trend='add',
            seasonal='add'
        )
        fitted_model = model.fit(
            smoothing_level=alpha,
            smoothing_slope=beta,
            smoothing_seasonal=gamma,
            optimized=False
        )

        # Forecast t+2 (second step ahead)
        forecast = fitted_model.forecast(2)[-1]  # Get the second step forecast
        t_plus_2_predictions.append(forecast)

        # Add the observed test value to the rolling training data using pd.concat()
        rolling_train = pd.concat([rolling_train, test.iloc[[i]]])

    # Combine the test dates with the predictions and actual sales into a DataFrame
    predictions_df = pd.DataFrame({
        'store_nbr': store_nbr,
        'item_nbr': item_nbr,
        'date': test_dates,
        'actual_sales': test['unit_sales'].values,
        't+2_prediction': t_plus_2_predictions
    })

    # Append the predictions for this item-store combination to the list
    all_predictions.append(predictions_df)

    # Calculate RMSE for this item-store combination
    rmse = np.sqrt(((predictions_df['actual_sales'] - predictions_df['t+2_prediction']) ** 2).mean())
    rmse_results.append({
        'store_nbr': store_nbr,
        'item_nbr': item_nbr,
        'RMSE': rmse
    })

# Combine all predictions into a single DataFrame
final_predictions_df = pd.concat(all_predictions, ignore_index=True)

# Convert RMSE results to a DataFrame for easier inspection
rmse_df = pd.DataFrame(rmse_results)

# Display the RMSE results for each item-store combination
print(rmse_df)

# Display the final predictions DataFrame (including predictions for each item-store combination)
print(final_predictions_df)


/var/folders/4z/hxnkb6_x7jn5rtztbhslhxdm0000gn/T/ipykernel_1595/3283468670.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['date'] = pd.to_datetime(filtered_df['date'])
/Users/Georgi/Documents/Group4B/venv_case_project/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-MON will be used.
  self._init_dates(dates, freq)
/var/folders/4z/hxnkb6_x7jn5rtztbhslhxdm0000gn/T/ipykernel_1595/3283468670.py:44: FutureWarning: the 'smoothing_slope' keyword is deprecated, use 'smoothing_trend' instead.
  fitted_model = model.fit(
/var/folders/4z/hxnkb6_x7jn5rtztbhslhxdm0000gn/T/ipykernel_1595/3283468670.py:52: FutureWarning: Series.__getitem__ treating 

     store_nbr  item_nbr       RMSE
0            1    105857  14.449060
1            1    106716   5.992163
2            1    108079  13.492055
3            1    108797  11.425733
4            1    111223  16.207059
..         ...       ...        ...
363          1   1317096   8.237393
364          1   1321497   9.896844
365          1   1324591  12.466696
366          1   1324667  17.689491
367          1   1324670   4.685522

[368 rows x 3 columns]
      store_nbr  item_nbr       date  actual_sales  t+2_prediction
0             1    105857 2016-08-15     39.000000       57.927869
1             1    105857 2016-08-22     32.000000       39.920853
2             1    105857 2016-08-29     31.714287       33.573831
3             1    105857 2016-09-05     30.333334       24.741926
4             1    105857 2016-09-12     38.666668       28.482757
...         ...       ...        ...           ...             ...
9563          1   1324670 2017-01-09      8.366667        3.781902
9564    

/var/folders/4z/hxnkb6_x7jn5rtztbhslhxdm0000gn/T/ipykernel_1595/3283468670.py:44: FutureWarning: the 'smoothing_slope' keyword is deprecated, use 'smoothing_trend' instead.
  fitted_model = model.fit(
/var/folders/4z/hxnkb6_x7jn5rtztbhslhxdm0000gn/T/ipykernel_1595/3283468670.py:52: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  forecast = fitted_model.forecast(2)[-1]  # Get the second step forecast
/Users/Georgi/Documents/Group4B/venv_case_project/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-MON will be used.
  self._init_dates(dates, freq)
/var/folders/4z/hxnkb6_x7jn5rtztbhslhxdm0000gn/T/ipykernel_1595/3283468670.py:44: FutureWarning: the 'smoothing_slope' keyword is deprecated, use 'smoothing_trend' in

In [17]:
# Calculate the average RMSE across all item-store combinations
average_rmse = np.mean([result['RMSE'] for result in rmse_results])
# Display the average RMSE
print("\nAverage RMSE across all item-store combinations:", average_rmse)


Average RMSE across all item-store combinations: 19.33809365957727
